In [7]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
import warnings # Ignore specific warnings
warnings.filterwarnings("ignore")

In [8]:
df1 = pd.read_excel('output_lube_oil_g11.xlsx')

In [9]:
# انتخاب ستون‌ها برای استانداردسازی
data_to_scale = df1[['AssetID_8341', 'AssetID_8342', 'AssetID_8343', 'AssetID_8344',
       'AssetID_8346', 'AssetID_9286', 'AssetID_9287']]

# استانداردسازی داده‌ها
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data_to_scale)

# تبدیل خروجی به دیتافریم با همان نام ستون‌ها
scaled_df = pd.DataFrame(scaled_data, columns=['AssetID_8341', 'AssetID_8342', 'AssetID_8343', 'AssetID_8344',
       'AssetID_8346', 'AssetID_9286', 'AssetID_9287'])


In [10]:
scaled_df_clean = scaled_df.dropna()

In [11]:
# اجرای DBSCAN
dbscan = DBSCAN(eps=0.7, min_samples=6)
labels = dbscan.fit_predict(scaled_df_clean)

# اضافه کردن لیبل‌ها به دیتافریم
df = scaled_df_clean.copy()
df['label'] = labels

# جدا کردن داده‌های نویز و خوشه‌ها
noise_mask = df['label'] == -1
cluster_mask = df['label'] != -1

noise_points = df[noise_mask].drop(columns='label').values
cluster_points = df[cluster_mask].drop(columns='label').values

# محاسبه فاصله هر نویز از نزدیک‌ترین نقطه در خوشه‌ها
distances = pairwise_distances(noise_points, cluster_points)
min_distances = distances.min(axis=1)

# نرمال‌سازی فاصله‌ها به بازه 0 تا 1
normalized_weights = (min_distances - min_distances.min()) / (min_distances.max() - min_distances.min())

# ساخت سری وزن ناهنجاری برای همه داده‌ها
anomaly_weights = np.zeros(len(df))
anomaly_weights[noise_mask.values] = normalized_weights

# اضافه کردن به دیتافریم نهایی
df['anomaly_weight'] = anomaly_weights


In [1]:
# train_model.py
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
import joblib
import warnings

warnings.filterwarnings("ignore")

# بارگذاری داده‌ها
df1 = pd.read_excel('output_lube_oil_g11.xlsx')
selected_columns = ['AssetID_8341', 'AssetID_8342', 'AssetID_8343', 'AssetID_8344',
                    'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
data_to_scale = df1[selected_columns]

# استانداردسازی
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data_to_scale)
scaled_df = pd.DataFrame(scaled_data, columns=selected_columns)
scaled_df_clean = scaled_df.dropna()

# اجرای DBSCAN
dbscan = DBSCAN(eps=0.7, min_samples=6)
labels = dbscan.fit_predict(scaled_df_clean)

# جدا کردن داده‌های نویز و خوشه‌ها
df = scaled_df_clean.copy()
df['label'] = labels
noise_mask = df['label'] == -1
cluster_mask = df['label'] != -1
noise_points = df[noise_mask].drop(columns='label').values
cluster_points = df[cluster_mask].drop(columns='label').values

# محاسبه فاصله و وزن ناهنجاری
distances = pairwise_distances(noise_points, cluster_points)
min_distances = distances.min(axis=1)
normalized_weights = (min_distances - min_distances.min()) / (min_distances.max() - min_distances.min())

# ساخت بردار وزن ناهنجاری
anomaly_weights = np.zeros(len(df))
anomaly_weights[noise_mask.values] = normalized_weights
df['anomaly_weight'] = anomaly_weights

# ذخیره مدل‌ها و داده‌های مرجع
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(dbscan, 'dbscan_model.pkl')
np.save('cluster_points.npy', cluster_points)


In [2]:
# predict_anomaly.py
import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances
import joblib

# بارگذاری اجزای مدل
scaler = joblib.load('scaler.pkl')
dbscan = joblib.load('dbscan_model.pkl')
cluster_points = np.load('cluster_points.npy')

# تابع تشخیص ناهنجاری
def is_anomalous(input_dict):
    input_df = pd.DataFrame([input_dict])
    scaled_input = scaler.transform(input_df)
    label = dbscan.fit_predict(scaled_input)[0]

    if label != -1:
        return {'is_anomaly': False, 'anomaly_weight': 0.0}

    # محاسبه فاصله از نزدیک‌ترین خوشه
    distance = pairwise_distances(scaled_input, cluster_points).min()
    # نرمال‌سازی با فرض بازه [0, 1] از مدل اصلی
    # برای دقت بیشتر می‌توان min و max را نیز ذخیره کرد
    anomaly_weight = distance  # یا نرمال‌سازی با مقادیر ذخیره‌شده
    return {'is_anomaly': True, 'anomaly_weight': anomaly_weight}

# مثال استفاده
sample_input = {
    'AssetID_8341': 12.5,
    'AssetID_8342': 8.3,
    'AssetID_8343': 5.1,
    'AssetID_8344': 9.7,
    'AssetID_8346': 6.2,
    'AssetID_9286': 4.8,
    'AssetID_9287': 7.9
}

result = is_anomalous(sample_input)
print(result)


{'is_anomaly': True, 'anomaly_weight': 121.52566629237832}
